Cell 1 — Imports

In [ ]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

Cell 2 — Paths

In [ ]:
# =========================
# Path configuration
# =========================
USE_GRID = False

# LOCAL_ROOT = Path("/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/")
LOCAL_ROOT = Path("/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext")
# LOCAL_ROOT = Path("/home/arsalan/wsu_grid_rs_grp_mljet/")

GRID_ROOT  = Path("/rs/rs_grp_mljet/ml-jet-ext/")

BASE_ROOT = GRID_ROOT if USE_GRID else LOCAL_ROOT

# Unfolded single-image dataset
UNFOLDED_ROOT = BASE_ROOT / "pTHat50-250_eCM5020_c0005_pT90-110_R0p4"

# Existing agg500 mean dataset (for consistency checks only)
OLD_AGG500_ROOT = BASE_ROOT / "pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500"

# New 5-moment dataset root
NEW_MOMENT_ROOT = BASE_ROOT / "pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500_m1-5_normalized"
NEW_MOMENT_METADATA_DIR = NEW_MOMENT_ROOT / "metadata"

NEW_MOMENT_ROOT.mkdir(parents=True, exist_ok=True)
NEW_MOMENT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("UNFOLDED_ROOT:", UNFOLDED_ROOT)
print("OLD_AGG500_ROOT:", OLD_AGG500_ROOT)
print("NEW_MOMENT_ROOT:", NEW_MOMENT_ROOT)
print("NEW_MOMENT_METADATA_DIR:", NEW_MOMENT_METADATA_DIR)

Cell 3 — Read the 35 config CSVs from the new metadata folder

In [ ]:
config_csvs = sorted(NEW_MOMENT_METADATA_DIR.glob("c0005_*_agg500_metadata.csv"))

print(f"Found {len(config_csvs)} config CSVs")
for p in config_csvs[:5]:
    print(" -", p.name)

Cell 4 — Helper: parse source file paths

In [ ]:
def parse_src_paths(value):
    """
    Supports:
      - Python list string
      - JSON list string
      - pipe-separated string
      - already a list
    """
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    s = str(value).strip()

    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception:
            try:
                parsed = json.loads(s)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed]
            except Exception:
                pass

    if "|" in s:
        return [x.strip() for x in s.split("|") if x.strip()]

    return [s]

Cell 5 — Infer metadata column names

In [ ]:
def find_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}")


def infer_metadata_columns(df):
    file_col = find_column(df, ["file_paths", "src_file_paths", "source_file_paths", "member_file_paths"])
    agg_col  = find_column(df, ["agg_file_path", "file_path", "agg_path"])

    alpha_col = find_column(df, ["alpha", "alpha_s", "alpha_value"])
    q0_col    = find_column(df, ["q0", "Q0", "q_0"])
    # eloss_col = find_column(df, ["energy_loss", "eloss", "energy_loss_module"])

    # sample_id_col = None
    # for c in ["agg_id", "sample_id", "group_id", "id"]:
    #     if c in df.columns:
    #         sample_id_col = c
    #         break

    return {
        "file_col": file_col,
        "agg_col": agg_col,
        "alpha_col": alpha_col,
        "q0_col": q0_col,
        # "eloss_col": eloss_col,
        # "sample_id_col": sample_id_col,
    }

Cell 6 — Inspect one config CSV

In [ ]:
test_csv = config_csvs[0]
df_test = pd.read_csv(test_csv)

print("Testing:", test_csv.name)
print("Shape:", df_test.shape)
display(df_test.head())

colmap = infer_metadata_columns(df_test)
print(colmap)

Cell 7 — Helper: load one unfolded jet image

In [ ]:
def load_single_jet_image(rel_path):
    rel_path = str(rel_path).strip()
    img_path = UNFOLDED_ROOT / rel_path

    if not img_path.exists():
        raise FileNotFoundError(f"Missing source image: {img_path}")

    arr = np.load(img_path).astype(np.float32)
    return arr

Cell 8 — Compute moments 1–5

In [ ]:
def compute_moments_1_to_5(stack):
    """
    stack: [N, 32, 32]
    returns: [5, 32, 32]
    """
    stack = np.asarray(stack, dtype=np.float32)
    assert stack.ndim == 3, f"Expected [N,H,W], got {stack.shape}"

    m1 = stack.mean(axis=0)
    centered = stack - m1

    m2 = (centered ** 2).mean(axis=0)
    m3 = (centered ** 3).mean(axis=0)
    m4 = (centered ** 4).mean(axis=0)
    m5 = (centered ** 5).mean(axis=0)

    return np.stack([m1, m2, m3, m4, m5], axis=0).astype(np.float32)

Cell 9 — Build the new output relative path
This preserves the same config-based folder style.

In [ ]:
def build_new_relpath_from_old_agg(old_agg_relpath: str):
    """
    Reuse the same relative structure as agg500 mean dataset,
    but save the new 5-channel tensor under the new root.
    """
    return str(old_agg_relpath)

Cell 10 — Smoke test on one aggregated sample

In [ ]:

row = df_test.iloc[0]

src_paths = parse_src_paths(row[colmap["file_col"]])
print("Number of source paths:", len(src_paths))
print("First 3 paths:")
for p in src_paths[:3]:
    print("  ", p)

stack = np.stack([load_single_jet_image(p) for p in src_paths], axis=0)
print("Stack shape:", stack.shape)

moments = compute_moments_1_to_5(stack)
print("Moments shape:", moments.shape)

for i in range(5):
    print(
        f"m{i+1}: "
        f"min={moments[i].min():.6f}, "
        f"max={moments[i].max():.6f}, "
        f"mean={moments[i].mean():.6f}, "
        f"std={moments[i].std():.6f}"
    )



Cell 12 — Visualize one sample

In [ ]:

titles = [
    "Moment 1: Mean",
    "Moment 2: Variance",
    "Moment 3",
    "Moment 4",
    "Moment 5"
]

fig, axes = plt.subplots(1, 5, figsize=(18, 3), dpi=150)

for i in range(5):
    ax = axes[i]
    im = ax.imshow(moments[i].T, origin="lower", cmap="inferno", aspect="auto")
    ax.set_title(titles[i])
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()



In [ ]:
import math


def log_normalize_image(img, epsilon=1e-8, img_max=1.0):
    """
    Log normalize nonnegative image to [0, 1].
    """
    I = np.asarray(img, dtype=np.float32)

    if img_max <= 0:
        return np.zeros_like(I, dtype=np.float32)

    num = np.log(I + epsilon) - math.log(epsilon)
    den = math.log(img_max + epsilon) - math.log(epsilon)

    if den <= 0:
        return np.zeros_like(I, dtype=np.float32)

    I_log = num / den
    return np.clip(I_log, 0.0, 1.0).astype(np.float32)


def signed_log_normalize_image(img, epsilon=1e-8, img_abs_max=1.0):
    """
    Signed log normalize image to [-1, 1].
    Suitable for odd central moments that may be negative.
    """
    I = np.asarray(img, dtype=np.float32)

    if img_abs_max <= 0:
        return np.zeros_like(I, dtype=np.float32)

    mag = np.abs(I)
    num = np.log(mag + epsilon) - math.log(epsilon)
    den = math.log(img_abs_max + epsilon) - math.log(epsilon)

    if den <= 0:
        return np.zeros_like(I, dtype=np.float32)

    I_log = num / den
    I_signed = np.sign(I) * np.clip(I_log, 0.0, 1.0)
    return I_signed.astype(np.float32)


def normalize_moments_1_to_5(moments, epsilon=1e-8):
    """
    moments: [5, H, W]
    Returns normalized moments:
      m1, m2, m4 -> [0, 1]
      m3, m5     -> [-1, 1]
    """
    moments = np.asarray(moments, dtype=np.float32)
    assert moments.shape[0] == 5, f"Expected 5 channels, got {moments.shape}"

    m1_max = 110.0
    m2_max = 110.0 ** 2
    m3_abs_max = 110.0 ** 3
    m4_max = 110.0 ** 4
    m5_abs_max = 110.0 ** 5

    out = np.stack([
        log_normalize_image(moments[0], epsilon=epsilon, img_max=m1_max),
        log_normalize_image(moments[1], epsilon=epsilon, img_max=m2_max),
        signed_log_normalize_image(moments[2], epsilon=epsilon, img_abs_max=m3_abs_max),
        log_normalize_image(moments[3], epsilon=epsilon, img_max=m4_max),
        signed_log_normalize_image(moments[4], epsilon=epsilon, img_abs_max=m5_abs_max),
    ], axis=0).astype(np.float32)

    return out


In [ ]:

moments = normalize_moments_1_to_5(moments, epsilon=1e-8)
print("Log-normalized moments shape:", moments.shape)
for i in range(5):
    print(
        f"m{i+1}_log: "
        f"min={moments[i].min():.6f}, "
        f"max={moments[i].max():.6f}, "
        f"mean={moments[i].mean():.6f}, "
        f"std={moments[i].std():.6f}"
    )


In [ ]:
        
titles = [
    "Moment 1: Mean",
    "Moment 2: Variance",
    "Moment 3",
    "Moment 4",
    "Moment 5"
]

fig, axes = plt.subplots(1, 5, figsize=(18, 3), dpi=150)

for i in range(5):
    ax = axes[i]
    im = ax.imshow(moments[i].T, origin="lower", cmap="inferno", aspect="auto")
    ax.set_title(titles[i])
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()



Cell 13 — Save one sample in the new format

In [ ]:

new_rel = build_new_relpath_from_old_agg(row[colmap["agg_col"]])
new_abs = NEW_MOMENT_ROOT / new_rel
new_abs.parent.mkdir(parents=True, exist_ok=True)

np.save(new_abs, moments.astype(np.float32))

print("Saved:", new_abs)
print("Saved shape:", np.load(new_abs).shape)



Cell 14 — Process one config CSV

In [ ]:
def process_one_config_csv(csv_path, overwrite=False, verbose=True):
    df = pd.read_csv(csv_path)
    colmap = infer_metadata_columns(df)

    out_rows = []

    if verbose:
        print(f"[INFO] Processing {csv_path.name} with {len(df)} rows")

    for _, row in tqdm(df.iterrows(), total=len(df), desc=csv_path.stem):
        src_paths = parse_src_paths(row[colmap["file_col"]])

        if len(src_paths) != 500:
            raise ValueError(
                f"{csv_path.name}: Expected 500 source images, got {len(src_paths)}"
            )

        stack = np.stack([load_single_jet_image(p) for p in src_paths], axis=0)
        moments = compute_moments_1_to_5(stack)
        moments_log = normalize_moments_1_to_5(moments, epsilon=1e-8)
        old_agg_rel = row[colmap["agg_col"]]
        new_rel = build_new_relpath_from_old_agg(old_agg_rel)
        new_abs = NEW_MOMENT_ROOT / new_rel
        new_abs.parent.mkdir(parents=True, exist_ok=True)

        if overwrite or (not new_abs.exists()):
            np.save(new_abs, moments_log.astype(np.float32))

        out_row = row.to_dict()
        out_row[colmap["agg_col"]] = new_rel
        out_rows.append(out_row)

    out_df = pd.DataFrame(out_rows)

    out_csv = NEW_MOMENT_METADATA_DIR / csv_path.name
    out_df.to_csv(out_csv, index=False)

    if verbose:
        print(f"[INFO] Saved updated metadata to: {out_csv}")
        print(f"[INFO] Wrote {len(out_df)} rows")

    return out_df

Cell 15 — Smoke test on one full config

In [ ]:
smoke_df = process_one_config_csv(config_csvs[0], overwrite=False, verbose=True)
display(smoke_df.head())

Cell 16 — Verify one saved file from the smoke run

In [ ]:
def verify_return_moments(config_index,image_index=500):
    smoke_csv = NEW_MOMENT_METADATA_DIR / config_csvs[config_index].name
    smoke_df = pd.read_csv(smoke_csv)
    smoke_colmap = infer_metadata_columns(smoke_df)

    sample_rel = smoke_df.iloc[image_index][smoke_colmap["agg_col"]]
    sample_abs = NEW_MOMENT_ROOT / sample_rel
    sample_arr = np.load(sample_abs)

    print("Sample file:", sample_abs)
    print("Shape:", sample_arr.shape)

    for i in range(5):
        print(
            f"m{i+1}: min={sample_arr[i].min():.6f}, "
            f"max={sample_arr[i].max():.6f}, "
            f"mean={sample_arr[i].mean():.6f}, "
            f"std={sample_arr[i].std():.6f}"
        )
    return sample_arr

config_sample_arr = []
for idx in range(len(config_csvs)):
    print(f"Verifying config {idx}: {config_csvs[idx].name}")
    sample_arr = verify_return_moments(config_index=idx, image_index=500)
    config_sample_arr.append(sample_arr)

In [ ]:
def visualize_moments(moments, titles = [
        "Moment 1: Mean",
        "Moment 2: Variance",
        "Moment 3",
        "Moment 4",
        "Moment 5"
    ]):
    

    fig, axes = plt.subplots(1, 5, figsize=(18, 3), dpi=150)

    for i in range(5):
        ax = axes[i]
        im = ax.imshow(moments[i].T, origin="lower", cmap="inferno", aspect="auto")
        ax.set_title(titles[i])
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

for idx, sample_arr in enumerate(config_sample_arr):
    # print(f"Visualizing config {idx}: {config_csvs[idx].name}")
    visualize_moments(sample_arr)
    # if idx == 0:  # Only visualize the first config for now
    #     visualize_moments(sample_arr)
    # else:
    #     visualize_moments(sample_arr, titles=None)



In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
DATASET_ROOT = Path("/mnt/Research/arsalan/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500_m1-5_normalized")

ALPHAS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
Q0S    = [1.0, 1.5, 2.0, 2.5, 3.0]

def fmt_alpha(a):
    return f"alpha{a:.2f}".replace(".", "p")

def fmt_q0(q):
    return f"q_{q:.1f}".replace(".", "p")

def config_dir_name(alpha, q0):
    return f"c0005_{fmt_alpha(alpha)}_{fmt_q0(q0)}"

def first_npy_in_dir(config_dir: Path):
    files = sorted(config_dir.glob("*.npy"))
    if not files:
        raise FileNotFoundError(f"No .npy files found in {config_dir}")
    return files[0]

# ------------------------------------------------------------
# Load one representative sample per config
# ------------------------------------------------------------
def load_representative_samples(dataset_root=DATASET_ROOT):
    """
    Returns:
        samples[(alpha, q0)] = np.ndarray of shape (5, 32, 32)
    """
    samples = {}
    for alpha in ALPHAS:
        for q0 in Q0S:
            cfg = config_dir_name(alpha, q0)
            cfg_dir = dataset_root / cfg
            sample_path = first_npy_in_dir(cfg_dir)
            arr = np.load(sample_path)
            if arr.shape != (5, 32, 32):
                raise ValueError(f"Unexpected shape {arr.shape} in {sample_path}")
            samples[(alpha, q0)] = arr
    return samples

# ------------------------------------------------------------
# Figure 1 / Supplementary grids: one moment over all configs
# ------------------------------------------------------------
def plot_moment_grid(samples, moment_idx, figsize=(10, 13), cmap="inferno",
                     same_scale=True, save_path=None, title=None):
    """
    moment_idx: 0-based index for M1..M5
    rows = alpha values
    cols = q0 values
    """
    nrows = len(ALPHAS)
    ncols = len(Q0S)

    # gather values for consistent scaling
    all_imgs = [samples[(a, q)][moment_idx] for a in ALPHAS for q in Q0S]
    if same_scale:
        vmin = min(img.min() for img in all_imgs)
        vmax = max(img.max() for img in all_imgs)
    else:
        vmin = vmax = None

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=200)

    for i, alpha in enumerate(ALPHAS):
        for j, q0 in enumerate(Q0S):
            ax = axes[i, j]
            img = samples[(alpha, q0)][moment_idx]

            im = ax.imshow(
                img.T,
                origin="lower",
                cmap=cmap,
                aspect="equal",
                vmin=vmin,
                vmax=vmax
            )

            ax.set_xticks([])
            ax.set_yticks([])

            if i == 0:
                ax.set_title(rf"$Q_0={q0:.1f}$", fontsize=10)
            if j == 0:
                ax.set_ylabel(rf"$\alpha_s={alpha:.2f}$", fontsize=10)

    if title is not None:
        fig.suptitle(title, fontsize=14, y=0.995)

    cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01)
    cbar.ax.tick_params(labelsize=9)

    plt.tight_layout(rect=[0, 0, 0.96, 0.985])

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

# ------------------------------------------------------------
# Figure 2: one sample across all 5 moments
# ------------------------------------------------------------
def plot_all_moments_of_sample(sample_arr, figsize=(18, 3.5), save_path=None, suptitle=None):
    titles = [
        "Moment 1: Mean",
        "Moment 2: Variance",
        "Moment 3",
        "Moment 4",
        "Moment 5"
    ]

    fig, axes = plt.subplots(1, 5, figsize=figsize, dpi=200)

    for i in range(5):
        ax = axes[i]

        # optional colormap choice
        cmap = "inferno"
        # If you want signed moments visually emphasized:
        # cmap = "RdBu_r" if i in [2, 4] else "inferno"

        img = sample_arr[i]
        im = ax.imshow(img.T, origin="lower", cmap=cmap, aspect="equal")
        ax.set_title(titles[i], fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=13, y=1.03)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
samples = load_representative_samples()

# Main-paper Figure A: Moment 1 over all configurations
plot_moment_grid(
    samples,
    moment_idx=0,
    figsize=(9, 12),
    cmap="inferno",
    save_path="figure_main_moment1_grid.png",
    title="Representative aggregate samples across all physics configurations (Moment 1)"
)

# Supplementary Figures S1-S4
for k in range(1, 5):
    plot_moment_grid(
        samples,
        moment_idx=k,
        figsize=(9, 12),
        cmap="inferno",
        save_path=f"figure_supp_moment{k+1}_grid.png",
        title=f"Representative aggregate samples across all physics configurations (Moment {k+1})"
    )

# Main-paper Figure B: one sample shown across all 5 moments
example_alpha = 0.20
example_q0 = 1.0
sample_arr = samples[(example_alpha, example_q0)]

plot_all_moments_of_sample(
    sample_arr,
    save_path="figure_main_single_sample_moments.png",
    suptitle=rf"One representative aggregate sample across moments ($\alpha_s={example_alpha:.2f}, Q_0={example_q0:.1f}$)"
)